<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/playfair_cipher_6x6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playfair Cipher

## History
The Playfair Cipher was invented by Charles Wheatstone in 1854, but it is named after Lord Playfair, who promoted its use to the British government. It was actually used by the British army in the Boer War and in World War 1, because it was fast enough to use by hand in the field, and much stronger than a simple substitution cipher.

## What is Playfair Cipher?
Playfair Cipher is different from every cipher we have looked at so far, because it does not encrypt one letter at a time. It encrypts **pairs of letters**, called **digraphs**. This makes simple frequency analysis on single letters useless against it, because the cipher is not hiding the frequency of A, B, C individually, it is hiding the frequency of letter pairs.

The classic textbook version of this cipher uses a **5x5 grid**, which only has room for 25 cells. Since the English alphabet has 26 letters, the classic version squeezes I and J into one shared cell, so a J in the message gets treated as an I. The problem with that is real: once you decrypt, you can never tell if the original letter was really I or really J, the information is permanently lost.

**This notebook fixes that problem.** Instead of a 5x5 grid, we use a **6x6 grid**, which has 36 cells. That is exactly enough room for all 26 letters (I and J both get their own cell) plus the 10 digits 0 to 9. So this version can encrypt names like Joy correctly, and it can also encrypt numbers directly, without needing to spell them out as words.

## Cryptography Algorithm

### 1. Building the Key Square
1.  Pick a keyword, for example **MONARCHY**.
2.  Write the keyword into the grid, left to right, top to bottom, skipping any letter that has already been used.
3.  Fill in the rest of the grid with the remaining letters of the alphabet, followed by the digits 0 to 9, also skipping repeats. Every letter, including both I and J, gets its own cell.

For the keyword **MONARCHY**, the 6x6 key square looks like this:

| | | | | | |
|---|---|---|---|---|---|
| M | O | N | A | R | C |
| H | Y | B | D | E | F |
| G | I | J | K | L | P |
| Q | S | T | U | V | W |
| X | Z | 0 | 1 | 2 | 3 |
| 4 | 5 | 6 | 7 | 8 | 9 |

### 2. Preparing the Plaintext
Before encryption, the plaintext must be cleaned and split into pairs, using these rules:

1.  Remove all spaces and punctuation. Keep only letters and digits.
2.  There is no need to merge J into I anymore, both have their own cell in the 6x6 grid.
3.  Split the letters into pairs, from left to right.
4.  If both characters in a pair are the same, insert an **X** between them, and re-pair the rest of the message.
5.  If the very last character is left alone with no partner, pad it with an **X**.

For example, **HELLO** becomes **HE LX LO**, because the double L needs an X inserted between them. And **JOY** stays exactly as **JOY**, it is no longer turned into IOY.

### 3. Encryption Rules
Every pair of characters is located inside the 6x6 key square, and one of three rules is applied, depending on where the two characters sit relative to each other.

**Rule 1: Same Row**
If both characters are in the same row, replace each one with the character immediately to its right, wrapping around to the start of the row if needed (the row only has 6 columns now, instead of 5).

**Rule 2: Same Column**
If both characters are in the same column, replace each one with the character immediately below it, wrapping around to the top of the column if needed (the column only has 6 rows now, instead of 5).

**Rule 3: Rectangle**
If the letters are in different rows and different columns, they form the two opposite corners of a rectangle. Replace each letter with the letter that sits in its own row, but in the column of the other letter.

### 4. Decryption Rules
Decryption uses the exact same key square and the exact same three rules, just reversed.

*   **Same Row**: move each letter one step to the **left** instead of right.
*   **Same Column**: move each letter one step **up** instead of down.
*   **Rectangle**: exactly the same rule as encryption, since swapping columns twice brings you back to where you started.

### 5. Fully Worked Example (By Hand)
Using the 6x6 key square built from **MONARCHY** above.

**Example A: Same Row rule.** Encrypt the pair **MO**.
Both M and O sit in row 0, at columns 0 and 1. Moving each one step to the right:
*   M (row 0, col 0) becomes the letter at (row 0, col 1) = **O**
*   O (row 0, col 1) becomes the letter at (row 0, col 2) = **N**

So **MO encrypts to ON**.

**Example B: Same Column rule.** Encrypt the pair **MH**.
M sits at row 0, col 0. H sits at row 1, col 0. Same column, so we move each one step down:
*   M (row 0, col 0) becomes the letter at (row 1, col 0) = **H**
*   H (row 1, col 0) becomes the letter at (row 2, col 0) = **G**

So **MH encrypts to HG**.

**Example C: Rectangle rule.** Encrypt the pair **ME**.
M sits at row 0, col 0. E sits at row 1, col 4. Different row, different column, so this is a rectangle.
*   M stays in row 0, but moves to E's column (col 4), giving the letter at (row 0, col 4) = **R**
*   E stays in row 1, but moves to M's column (col 0), giving the letter at (row 1, col 0) = **H**

So **ME encrypts to RH**.

This is exactly the logic the code below runs automatically for every pair in a full message.

### 1. Import Dependencies

In [1]:
import random

### 2. Build the Key Square

In [2]:
PLAYFAIR_ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"  # 36 characters: full A-Z plus 0-9
GRID_SIZE = 6  # 6 x 6 = 36 cells, exactly matches PLAYFAIR_ALPHABET

def build_key_square(keyword: str) -> list:
    keyword = keyword.upper()

    seen = set()
    square = []

    # first place every unique character of the keyword
    for ch in keyword:
        if ch in PLAYFAIR_ALPHABET and ch not in seen:
            seen.add(ch)
            square.append(ch)

    # then fill in the rest of the alphabet and digits, in order
    # both I and J get their own cell here, nothing is merged
    for ch in PLAYFAIR_ALPHABET:
        if ch not in seen:
            seen.add(ch)
            square.append(ch)

    grid = [square[i * GRID_SIZE:(i + 1) * GRID_SIZE] for i in range(GRID_SIZE)]
    return grid

def find_position(grid: list, ch: str):
    for row in range(GRID_SIZE):
        for col in range(GRID_SIZE):
            if grid[row][col] == ch:
                return row, col
    return None

### 3. Prepare the Plaintext into Digraphs

In [3]:
def prepare_text(text: str) -> list:
    # keep letters and digits, since our 6x6 grid can handle both
    # I and J are no longer merged, each has its own cell
    cleaned = "".join(ch for ch in text.upper() if ch in PLAYFAIR_ALPHABET)

    digraphs = []
    i = 0
    while i < len(cleaned):
        first = cleaned[i]

        if i + 1 < len(cleaned):
            second = cleaned[i + 1]
            if first == second:
                # double characters need an X inserted between them
                digraphs.append(first + "X")
                i += 1
            else:
                digraphs.append(first + second)
                i += 2
        else:
            # last character with no partner gets padded with X
            digraphs.append(first + "X")
            i += 1

    return digraphs

### 4. Generate a Random Key

In [4]:
def generate_random_key() -> str:
    # shuffle all 36 characters (A-Z and 0-9) into a random order to build a random key square
    characters = list(PLAYFAIR_ALPHABET)
    random.shuffle(characters)
    return "".join(characters)

### 5. Encryption

In [5]:
def encrypt_pair(grid: list, a: str, b: str) -> str:
    row_a, col_a = find_position(grid, a)
    row_b, col_b = find_position(grid, b)

    if row_a == row_b:
        # same row: shift right, wrap around with modulo 6
        return grid[row_a][(col_a + 1) % GRID_SIZE] + grid[row_b][(col_b + 1) % GRID_SIZE]
    elif col_a == col_b:
        # same column: shift down, wrap around with modulo 6
        return grid[(row_a + 1) % GRID_SIZE][col_a] + grid[(row_b + 1) % GRID_SIZE][col_b]
    else:
        # rectangle: swap columns, keep each character's own row
        return grid[row_a][col_b] + grid[row_b][col_a]

def encrypt(text: str, keyword: str) -> str:
    grid = build_key_square(keyword)
    digraphs = prepare_text(text)
    return "".join(encrypt_pair(grid, pair[0], pair[1]) for pair in digraphs)

### 6. Decryption

In [6]:
def decrypt_pair(grid: list, a: str, b: str) -> str:
    row_a, col_a = find_position(grid, a)
    row_b, col_b = find_position(grid, b)

    if row_a == row_b:
        # same row: shift left, wrap around with modulo 6
        return grid[row_a][(col_a - 1) % GRID_SIZE] + grid[row_b][(col_b - 1) % GRID_SIZE]
    elif col_a == col_b:
        # same column: shift up, wrap around with modulo 6
        return grid[(row_a - 1) % GRID_SIZE][col_a] + grid[(row_b - 1) % GRID_SIZE][col_b]
    else:
        # rectangle: same swap as encryption
        return grid[row_a][col_b] + grid[row_b][col_a]

def decrypt(cipher_text: str, keyword: str) -> str:
    grid = build_key_square(keyword)
    pairs = [cipher_text[i:i + 2] for i in range(0, len(cipher_text), 2)]
    return "".join(decrypt_pair(grid, pair[0], pair[1]) for pair in pairs)

### 7. Verify the Hand Worked Example in Code

In [7]:
hand_keyword = "MONARCHY"
hand_grid = build_key_square(hand_keyword)

print("Key Square:")
for row in hand_grid:
    print(row)

print("\nMO ->", encrypt_pair(hand_grid, "M", "O"), "(should match ON from the hand example)")
print("MH ->", encrypt_pair(hand_grid, "M", "H"), "(should match HG from the hand example)")
print("ME ->", encrypt_pair(hand_grid, "M", "E"), "(should match RH from the hand example)")

joy_test = encrypt("JOY", hand_keyword)
joy_back = decrypt(joy_test, hand_keyword)
print(f"\nJOY encrypts to {joy_test}, and decrypts back to {joy_back} (J stays J, not I)")

Key Square:
['M', 'O', 'N', 'A', 'R', 'C']
['H', 'Y', 'B', 'D', 'E', 'F']
['G', 'I', 'J', 'K', 'L', 'P']
['Q', 'S', 'T', 'U', 'V', 'W']
['X', 'Z', '0', '1', '2', '3']
['4', '5', '6', '7', '8', '9']

MO -> ON (should match ON from the hand example)
MH -> HG (should match HG from the hand example)
ME -> RH (should match RH from the hand example)

JOY encrypts to INHZ, and decrypts back to JOYX (J stays J, not I)


### 8. Example usage

In [14]:
key = generate_random_key()
print(f"Generated Random Key: {key}")

Generated Random Key: 6HNBWTCS9MVP0R7EDIOKYA52GJLU148XZQF3


In [15]:
plaintext = "TOP secret Massage! Agent Joy, visit Area fifty-one"
print(f"Original Plain Text: {plaintext}")
print(f"Prepared Digraphs: {prepare_text(plaintext)}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

cleaned_digraph_text = "".join(prepare_text(plaintext))
match = cleaned_digraph_text == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent Joy, visit Area fifty-one
Prepared Digraphs: ['TO', 'PS', 'EC', 'RE', 'TM', 'AS', 'SA', 'GE', 'AG', 'EN', 'TJ', 'OY', 'VI', 'SI', 'TA', 'RE', 'AF', 'IF', 'TY', 'ON', 'EX']
Encrypted: 62C90M7DBPKMMKU0OU7BH4KAPDPRB27D5QD3N2Y6RQ
Decrypted: TOPSECRETMASSAGEAGENTJOYVISITAREAFIFTYONEX
Verification Match:True
